In [ ]:
import torch
from torch import nn
from torch import optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.nn import functional as F

data

In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True,
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
)

model

In [ ]:
#forward函数return logvar,mu
class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super().__init__()

        # 编码器：784 -> 400
        self.encoder = nn.Linear(28 * 28, 400)

        # 分别输出潜变量分布的均值和对数方差
        self.fc_mu = nn.Linear(400, latent_dim)
        self.fc_logvar = nn.Linear(400, latent_dim)

        # 解码器：latent_dim -> 400 -> 784
        self.decoder_input = nn.Linear(latent_dim, 400)
        self.decoder_output = nn.Linear(400, 28 * 28)

    def encode(self, x):
        x = x.flatten(start_dim=1)
        hidden = F.relu(self.encoder(x))

        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)

        return mu, logvar
    #重参数采样
    def reparameterize(self, mu, logvar):
        # std = sqrt(var) = exp(0.5 * logvar)
        std = torch.exp(0.5 * logvar)
        #epsilon从N(0,1)中取出
        epsilon = torch.randn_like(std)

        return mu + epsilon * std

    def decode(self, z):
        hidden = F.relu(self.decoder_input(z))
        reconstruction = torch.sigmoid(self.decoder_output(hidden))

        return reconstruction.view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z)

        return reconstruction, mu, logvar
def vae_loss(reconstruction, x, mu, logvar):
    # 1. 重建损失：衡量重建图片与原图片的差异
    #符合伯努利分布，经数学推导用交叉熵计算重建损失
    reconstruction_loss = F.binary_cross_entropy(
        reconstruction,
        x,
        reduction="sum",
    )

    # 2. KL 散度：让潜变量分布接近标准正态分布 N(0, 1)
    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    # 3. VAE 总损失
    total_loss = reconstruction_loss + kl_loss

    return total_loss, reconstruction_loss, kl_loss

train

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VAE(latent_dim=20).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
def train(model, train_loader, optimizer, device, epoch):
    model.train()

    total_loss = 0
    total_reconstruction_loss = 0
    total_kl_loss = 0

    for batch_index, (images, _) in enumerate(train_loader):
        images = images.to(device)

        # 清空上一次迭代的梯度
        optimizer.zero_grad()

        # 前向传播
        reconstruction, mu, logvar = model(images)

        # 计算损失
        loss, reconstruction_loss, kl_loss = vae_loss(
            reconstruction,
            images,
            mu,
            logvar,
        )

        # 反向传播并更新参数
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_reconstruction_loss += reconstruction_loss.item()
        total_kl_loss += kl_loss.item()

        if batch_index % 100 == 0:
            average_loss = loss.item() / images.size(0)
if __name__ == "__main__":
    epochs = 10

    for epoch in range(1, epochs + 1):
        train(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            device=device,
            epoch=epoch,
        )

test

In [ ]:
model.eval()
num_samples = 16
latent_dim = 20
with torch.no_grad():
    # 从标准正态分布 N(0, I) 中采样隐变量
    z = torch.randn(num_samples, latent_dim, device=device)

    # 通过 decoder 生成图片
    generated_images = model.decode(z).cpu()
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for image, ax in zip(generated_images, axes.flat):
    ax.imshow(image.squeeze(0), cmap="gray")
    ax.axis("off")
plt.tight_layout()
plt.show()